# Rediscover the Z boson

**A guided HEPLab masterclass · about 60 minutes**

Follow the analysis from event selection to the Z mass. The checkpoints give immediate feedback and progressive hints. Your progress in this notebook stays in the Python session; save a resumable record at the end.

> **Teaching sample:** the included events are deterministic synthetic data, not CMS collision data. The same lab accepts an authorized local CMS Open Data sample.

## Set up the lab

Install the notebook and analysis extras once. The controls below use `ipywidgets`; selecting an answer does not send your data anywhere.

In [ ]:
import heplab

lab = heplab.labs.start("z-boson")
lab.check

## Reproduce a Workbench run

Save the Workbench export as `heplab-z-boson-result.json` next to this notebook, or edit `export_path` below. Restart the kernel and run all cells. This section works without a network connection once HEPLab and the analysis environment are installed. It verifies the artifact checksum, recomputes the cutflow and mass histogram, and checks the mean and uncertainty of a converged fit to 0.005 GeV.

Without a Workbench export, continue to the guided investigation below.

In [ ]:
from pathlib import Path

from IPython.display import display
from heplab.labs.z_boson import reopen, reproduce

export_path = Path("heplab-z-boson-result.json")
workbench_run = reopen(export_path) if export_path.is_file() else None
if workbench_run is not None:
    replay_events, replay_result = reproduce(workbench_run)
    print(workbench_run["scope"])
    print("Reproduced the Workbench cutflow and mass histogram.")
    fit = workbench_run["fit"]
    if fit and fit["success"]:
        print(f'Fitted peak: {fit["values"]["mean"]:.3f} ± {fit["errors"]["mean"]:.3f} GeV')
    else:
        print("The exported run has no converged fit to compare.")
    display(heplab.notebook.show(replay_result))
else:
    print(f"Place the Workbench export at {export_path.resolve()} to reproduce it.")

### Change one cut

Predict what will happen to the cutflow and the tails of the mass distribution when you change the isolation threshold. The cell keeps all the other choices from your Workbench run.

In [ ]:
from copy import deepcopy

if workbench_run is not None:
    varied = deepcopy(workbench_run["analysis_config"])
    muons = varied["selection"]["muons"]
    original_iso = muons["max_rel_iso"]
    muons["max_rel_iso"] = 0.10 if original_iso > 0.10 else 0.30
    changed_result = heplab.run(replay_events, config=varied)
    print(f'Isolation {original_iso:g} → {muons["max_rel_iso"]:g}')
    for stage, baseline in replay_result["cutflow"].items():
        print(f'{stage}: {baseline} → {changed_result["cutflow"][stage]} events')
    print(f'Selected pairs: {replay_result["histogram"]["entries"]} → '
          f'{changed_result["histogram"]["entries"]}')
    display(heplab.notebook.show(changed_result))

## Your investigation

Work through the six checkpoints in order. You can ask for hints at any step. In the mass checkpoint, calculate the two muon four-vectors before entering your answer. In the fitting checkpoint, run `heplab.fit_peak(result)` in the analysis cell below and report its mean and uncertainty.

In [ ]:
lab_view = lab.display()
lab_view

## Inspect your measurement

After completing the checkpoints above, run this cell to create the analysis result and inspect its interactive histogram, cutflow, and provenance. The selected events remain in `lab._events` for this session.

In [ ]:
if lab._events is None or "config" not in lab.context:
    print("Complete the load and selection checkpoints above, then run this cell again.")
else:
    result = heplab.run(lab._events, config=lab.context["config"])
    fit = heplab.fit_peak(result)
    print(f"Fitted mass: {fit['values']['mean']:.2f} ± {fit['errors']['mean']:.2f} GeV")
    heplab.notebook.show(result)

## Keep your work

The interactive view saves progress in this notebook session. Save a portable checkpoint file with the events attached; resume it later with the same event list so event-dependent answers can be verified.

In [ ]:
if lab._events is not None:
    lab.save("z-boson-progress.json")
    print("Saved progress with an event-sample digest.")
else:
    print("Complete the load checkpoint before saving resumable progress.")

## Share a result

When `result` is available, export a self-contained HTML page. It includes the plot, provenance, and result record, and can be opened offline.

In [ ]:
if "result" in globals():
    heplab.export_html(result, "z-boson-result.html", title="Rediscover the Z boson")
else:
    print("Run the analysis cell above first.")